# Stock Market AI Advisor - Unified Model Training Pipeline
**Developed by**: Adithya Dadi  
**Program**: Summer Internship - Agentic AI | DataPro  

This notebook implements the complete, end-to-end, independent machine learning pipeline for the Stock Market AI Advisor. It handles:
1. Google Colab / Local environment setup.
2. **Automatic Kaggle Credentials Extraction**: Reads credentials directly from `kaggle.json` inside your user `Downloads` folder, Colab workspace, or local directory.
3. **Direct Kaggle Dataset Download**: Pulls and extracts `jacksoncrow/stock-market-dataset` directly from Kaggle.
4. **Self-Contained Data Merger**: Merges the Kaggle base files with live up-to-date market prices up to 2026 via `yfinance` entirely inside this notebook.
5. **Self-Contained Feature Engineering**: Computes advanced technical indicators (Moving Averages, RSI, MACD, Volatility, Price Ranges) natively.
6. **Model Training with Epochs & Optimization Logs**:
   - **Random Forest**: Hyperparameter grid search and tree building progress.
   - **XGBoost**: Detailed training loss across boosting rounds (epochs) on validation splits.
   - **Support Vector Machine (LinearSVR & SVC)**: Optimization solver step-by-step logs.
7. **Unified Performance Evaluation**: Cross-model performance comparison reports for regression and classification.

In [ ]:
# ==========================================================================
# GOOGLE COLAB / LOCAL WORKSPACE SETUP
# ==========================================================================
import sys
import os

if 'google.colab' in sys.modules:
    print("[INFO] Google Colab environment detected.")
    repo_url = 'https://github.com/Adi4224/Stock-market-AI-advisor.git'
    repo_name = 'Stock-market-AI-advisor'
    
    if not os.path.exists(repo_name):
        print(f"[INFO] Cloning project repository from {repo_url}...")
        !git clone {repo_url}
    else:
        print("[INFO] Project repository already exists in session files.")
        
    print(f"[INFO] Navigating into project directory: {repo_name}...")
    %cd {repo_name}
    print("[INFO] Working directory updated to:", os.getcwd())
else:
    print("[INFO] Running in local environment. Current working directory:", os.getcwd())

In [ ]:
# ==========================================================================
# AUTOMATIC KAGGLE CREDENTIALS EXTRACTION & SECURE DOWNLOAD
# ==========================================================================
import os
import json
import sys
import shutil

def configure_kaggle_credentials():
    raw_data_dir = 'data/raw/kaggle_stock_data/stocks'
    if os.path.exists(raw_data_dir) and len(os.listdir(raw_data_dir)) > 0:
        print("[INFO] Raw stock dataset already exists in storage. Skipping download.")
        return True
        
    print("[INFO] Searching for kaggle.json in standard user paths...")
    home_dir = os.path.expanduser('~')
    
    # List of possible locations where the user's kaggle.json may be downloaded
    candidate_paths = [
        os.path.join(home_dir, 'Downloads', 'kaggle.json'),
        os.path.join(home_dir, 'downloads', 'kaggle.json'),
        os.path.join(home_dir, 'OneDrive', 'Downloads', 'kaggle.json'),
        os.path.join(home_dir, 'OneDrive', 'downloads', 'kaggle.json'),
        os.path.join(home_dir, '.kaggle', 'kaggle.json'),
        '/content/kaggle.json', # Google Colab root folder
        'kaggle.json' # Current folder
    ]
    
    found_creds_path = None
    for path in candidate_paths:
        if os.path.exists(path):
            found_creds_path = path
            break
            
    if found_creds_path is None:
        print("[ERROR] kaggle.json not found in Downloads, Colab root, or default folders.")
        print("--- Quick Fix Required ---")
        print("Please download your Kaggle API token (kaggle.json) from Kaggle -> Settings -> Account -> Create New Token,")
        print("and save it inside your user Downloads folder:")
        print(f"  {os.path.join(home_dir, 'Downloads', 'kaggle.json')}")
        print("Then, re-run this cell.")
        return False
        
    print(f"[SUCCESS] Automatically detected kaggle.json at: {found_creds_path}")
    try:
        with open(found_creds_path, 'r') as f:
            creds = json.load(f)
            
        username = creds.get('username')
        api_key = creds.get('key')
        if not username or not api_key:
            print("[ERROR] Invalid format inside kaggle.json. Re-download your API token.")
            return False
            
        # Set environment variables for Kaggle CLI
        os.environ['KAGGLE_USERNAME'] = username
        os.environ['KAGGLE_KEY'] = api_key
        
        # Write to default ~/.kaggle/kaggle.json folder for CLI native integration
        kaggle_home_dir = os.path.join(home_dir, '.kaggle')
        os.makedirs(kaggle_home_dir, exist_ok=True)
        target_json_path = os.path.join(kaggle_home_dir, 'kaggle.json')
        
        with open(target_json_path, 'w') as f:
            json.dump(creds, f)
            
        if sys.platform != 'win32':
            os.chmod(target_json_path, 0o600)
        print(f"[INFO] Securely wrote Kaggle credentials to system profile: {target_json_path}")
        return True
    except Exception as e:
        print(f"[ERROR] Failed to load credentials from file: {e}")
        return False

if configure_kaggle_credentials():
    print("[INFO] Installing Kaggle CLI...")
    !pip install -q kaggle
    
    print("[INFO] Downloading stock-market-dataset from Kaggle API...")
    !kaggle datasets download -d jacksoncrow/stock-market-dataset
    
    if os.path.exists('stock-market-dataset.zip'):
        print("[INFO] Extracting dataset zip...")
        os.makedirs('data/raw/kaggle_stock_data', exist_ok=True)
        !unzip -q stock-market-dataset.zip -d data/raw/kaggle_stock_data/
        print("[SUCCESS] Dataset successfully extracted and saved to data/raw/kaggle_stock_data/")
        os.remove('stock-market-dataset.zip')
    else:
        print("[ERROR] Failed to download dataset. Check your Kaggle API key status.")

In [ ]:
# ==========================================================================
# SELF-CONTAINED DATA MERGER PIPELINE (KAGGLE + LIVE YFINANCE TO 2026)
# ==========================================================================
import os
import sys
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

print("[INFO] Installing yfinance and training dependencies...")
!pip install -q yfinance pandas numpy scikit-learn xgboost matplotlib joblib

import yfinance as yf

DEFAULT_TICKERS = [
    # Technology
    "AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "META", "NVDA", "AMD", "INTC", "CRM",
    "ADBE", "ORCL", "NFLX", "PYPL", "UBER",
    # Finance
    "JPM", "BAC", "GS", "MS", "WFC", "V", "MA", "AXP", "C", "BLK",
    # Healthcare
    "JNJ", "PFE", "UNH", "MRK", "ABT", "LLY", "ABBV", "TMO", "MDT", "BMY",
    # Energy
    "XOM", "CVX", "COP", "SLB", "EOG",
    # Consumer / Entertainment
    "WMT", "PG", "KO", "PEP", "COST", "HD", "NKE", "MCD", "SBUX", "DIS",
]

def normalise_columns(df):
    df.columns = [c.strip().title() for c in df.columns]
    if isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        
    df.rename(columns={"Adj Close": "Adj_Close", "Adj_close": "Adj_Close", "Adjclose": "Adj_Close"}, inplace=True)
    if "Date" in df.columns:
        dates = pd.to_datetime(df["Date"], errors="coerce")
        try:
            if dates.dt.tz is not None:
                dates = dates.dt.tz_localize(None)
        except AttributeError:
            pass
        df["Date"] = dates
        df.dropna(subset=["Date"], inplace=True)
    return df

def merge_and_update_all():
    kaggle_dir = 'data/raw/kaggle_stock_data/stocks'
    update_dir = 'data/updated/yfinance_updates'
    processed_dir = 'data/processed'
    
    os.makedirs(update_dir, exist_ok=True)
    os.makedirs(processed_dir, exist_ok=True)
    
    all_frames = []
    total = len(DEFAULT_TICKERS)
    
    print(f"[INFO] Beginning merge pipeline for {total} stock tickers...")
    for idx, ticker in enumerate(DEFAULT_TICKERS, start=1):
        kaggle_path = os.path.join(kaggle_dir, f"{ticker}.csv")
        old_df = pd.DataFrame()
        fetch_start = None
        
        # 1. Read existing Kaggle data if available
        if os.path.isfile(kaggle_path):
            try:
                old_df = pd.read_csv(kaggle_path)
                old_df = normalise_columns(old_df)
                if not old_df.empty and "Date" in old_df.columns:
                    last_date = old_df["Date"].max()
                    fetch_start = last_date + timedelta(days=1)
            except Exception as e:
                print(f"[{ticker}] Warning loading Kaggle base: {e}")
                
        # 2. Fetch incremental rows from yfinance
        try:
            yf_ticker = yf.Ticker(ticker)
            if fetch_start is not None:
                today_str = datetime.today().strftime("%Y-%m-%d")
                start_str = fetch_start.strftime("%Y-%m-%d")
                if fetch_start > datetime.today():
                    new_df = pd.DataFrame()
                else:
                    new_df = yf_ticker.history(start=start_str, end=today_str)
            else:
                # Fallback download of entire history if Kaggle folder is missing
                new_df = yf_ticker.history(period="max")
                
            if new_df is not None and not new_df.empty:
                new_df = normalise_columns(new_df)
            else:
                new_df = pd.DataFrame()
        except Exception as e:
            print(f"[{ticker}] yfinance download failed: {e}")
            new_df = pd.DataFrame()
            
        # 3. Concatenate and deduplicate old + new
        if old_df.empty and new_df.empty:
            continue
            
        merged = pd.concat([old_df, new_df], ignore_index=True)
        if "Date" in merged.columns:
            merged.drop_duplicates(subset=["Date"], keep="last", inplace=True)
            merged.sort_values("Date", inplace=True)
            merged.reset_index(drop=True, inplace=True)
            
        # Save individual ticker update
        out_path = os.path.join(update_dir, f"{ticker}.csv")
        merged.to_csv(out_path, index=False)
        
        # Append for combined database
        merged["Ticker"] = ticker
        all_frames.append(merged)
        
        if idx % 10 == 0 or idx == total:
            print(f"[INFO] Processed and merged {idx}/{total} stocks...")
            
    if all_frames:
        combined = pd.concat(all_frames, ignore_index=True)
        combined_path = os.path.join(processed_dir, 'final_stock_dataset_2026.csv')
        combined.to_csv(combined_path, index=False)
        print(f"[SUCCESS] Unified dataset created at: {combined_path} ({len(combined)} rows).")
    else:
        print("[ERROR] No data was combined. Check internet connection and dataset files.")

merge_and_update_all()

In [ ]:
# ==========================================================================
# SELF-CONTAINED TECHNICAL FEATURE ENGINEERING PIPELINE
# ==========================================================================
import pandas as pd
import numpy as np

data_path = 'data/processed/final_stock_dataset_2026.csv'
print(f"[INFO] Loading combined dataset from {data_path}...")
df = pd.read_csv(data_path)
print(f"[INFO] Dataset shape loaded: {df.shape}")

def compute_rsi(close, period=14):
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))

def build_technical_features(df):
    df = df.copy()
    result_frames = []
    
    # Calculate technical indicators independently per stock ticker
    for ticker, group in df.groupby('Ticker', sort=False):
        group = group.sort_values('Date').copy() if 'Date' in group.columns else group.copy()
        
        # Daily return
        group['daily_return'] = group['Close'].pct_change()
        
        # Moving averages
        group['ma_7'] = group['Close'].rolling(window=7, min_periods=1).mean()
        group['ma_14'] = group['Close'].rolling(window=14, min_periods=1).mean()
        group['ma_30'] = group['Close'].rolling(window=30, min_periods=1).mean()
        group['ma_50'] = group['Close'].rolling(window=50, min_periods=1).mean()
        
        # Rolling Volatility
        group['volatility'] = group['daily_return'].rolling(window=20, min_periods=1).std()
        
        # Relative Strength Index (RSI)
        group['rsi'] = compute_rsi(group['Close'], period=14)
        
        # MACD & Signal
        ema_12 = group['Close'].ewm(span=12, adjust=False).mean()
        ema_26 = group['Close'].ewm(span=26, adjust=False).mean()
        group['macd'] = ema_12 - ema_26
        group['macd_signal'] = group['macd'].ewm(span=9, adjust=False).mean()
        
        # Vol change
        group['volume_change'] = group['Volume'].pct_change()
        
        # High-Low Range ratio
        group['price_range'] = (group['High'] - group['Low']) / group['Close']
        
        # Shift to create the next-day close regression targets
        group['next_day_close'] = group['Close'].shift(-1)
        group['movement'] = (group['next_day_close'] > group['Close']).astype(int)
        
        result_frames.append(group)
        
    return pd.concat(result_frames, ignore_index=True)

print("[INFO] Calculating technical indicators and target variables...")
df = build_technical_features(df)

feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'daily_return', 'ma_7', 'ma_14', 'ma_30', 'ma_50',
    'volatility', 'rsi', 'macd', 'macd_signal', 'volume_change', 'price_range'
]

# Resolve potential infinite ratios and drop NA rows from rolling windows
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=feature_cols + ['next_day_close', 'movement']).reset_index(drop=True)

print(f"[SUCCESS] Technical feature matrix prepared: {df.shape[0]} rows, {len(feature_cols)} features.")

In [ ]:
# ==========================================================================
# TIME-SERIES SPLITTING AND SCALING PREPARATION
# ==========================================================================
from sklearn.preprocessing import StandardScaler

# 80-20 Chronological train-test split to avoid time-series forward lookahead leakage
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df[feature_cols].values
X_test = test_df[feature_cols].values

y_train_reg = train_df['next_day_close'].values
y_test_reg = test_df['next_day_close'].values
y_train_cls = train_df['movement'].values
y_test_cls = test_df['movement'].values

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Scale SVR target scaling for numerical stability
svr_target_scaler = StandardScaler()
y_train_reg_svr = svr_target_scaler.fit_transform(y_train_reg.reshape(-1, 1)).flatten()
y_test_reg_svr = svr_target_scaler.transform(y_test_reg.reshape(-1, 1)).flatten()

print(f"[INFO] Training Matrix: {X_train.shape} | Evaluation Matrix: {X_test.shape}")

In [ ]:
# ==========================================================================
# 1. MODEL TRAINING - RANDOM FOREST (WITH VERBOSE FITTING PROGRESS)
# ==========================================================================
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import joblib
import time

tscv = TimeSeriesSplit(n_splits=3)

print("[INFO] Tuning and training Random Forest Regressor...")
rf_reg_grid = {'n_estimators': [100, 200], 'max_depth': [10, 15]}
rf_reg_search = GridSearchCV(RandomForestRegressor(random_state=42), rf_reg_grid, cv=tscv, n_jobs=-1, verbose=1)
rf_reg_search.fit(X_train_scaled, y_train_reg)

print(f"[INFO] Best RF Regressor parameters: {rf_reg_search.best_params_}")
print("[INFO] Fitting final Random Forest Regressor and printing tree building progress...")
rf_reg_model = RandomForestRegressor(
    n_estimators=rf_reg_search.best_params_['n_estimators'],
    max_depth=rf_reg_search.best_params_['max_depth'],
    random_state=42,
    verbose=1
)
start_time = time.time()
rf_reg_model.fit(X_train_scaled, y_train_reg)
print(f"[SUCCESS] RF Regressor trained in {time.time() - start_time:.2f} seconds.")

print("\n[INFO] Tuning and training Random Forest Classifier...")
rf_cls_grid = {'n_estimators': [100, 200], 'max_depth': [10, 15]}
rf_cls_search = GridSearchCV(RandomForestClassifier(random_state=42), rf_cls_grid, cv=tscv, n_jobs=-1, verbose=1)
rf_cls_search.fit(X_train_scaled, y_train_cls)

print(f"[INFO] Best RF Classifier parameters: {rf_cls_search.best_params_}")
print("[INFO] Fitting final Random Forest Classifier and printing tree building progress...")
rf_cls_model = RandomForestClassifier(
    n_estimators=rf_cls_search.best_params_['n_estimators'],
    max_depth=rf_cls_search.best_params_['max_depth'],
    random_state=42,
    verbose=1
)
start_time = time.time()
rf_cls_model.fit(X_train_scaled, y_train_cls)
print(f"[SUCCESS] RF Classifier trained in {time.time() - start_time:.2f} seconds.")

os.makedirs('models', exist_ok=True)
joblib.dump(rf_reg_model, 'models/random_forest_regressor.pkl')
joblib.dump(rf_cls_model, 'models/random_forest_classifier.pkl')
print("[INFO] Random Forest models saved successfully inside models/")

In [ ]:
# ==========================================================================
# 2. MODEL TRAINING - XGBOOST (WITH DETAILED FITTING PROGRESS EPOCHS)
# ==========================================================================
import xgboost as xgb

print("[INFO] Tuning and training XGBoost Regressor...")
xgb_reg_grid = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]}
xgb_reg_search = GridSearchCV(xgb.XGBRegressor(objective='reg:squarederror', random_state=42), xgb_reg_grid, cv=tscv, n_jobs=-1, verbose=1)
xgb_reg_search.fit(X_train_scaled, y_train_reg)

print(f"[INFO] Best XGBoost Regressor parameters: {xgb_reg_search.best_params_}")
print("[INFO] Fitting final XGBoost Regressor and displaying training rounds (epochs) and losses live...")
xgb_reg_model = xgb.XGBRegressor(
    n_estimators=xgb_reg_search.best_params_['n_estimators'],
    learning_rate=xgb_reg_search.best_params_['learning_rate'],
    objective='reg:squarederror',
    random_state=42
)

# Fit with evaluation sets to show training epochs progress live
xgb_reg_model.fit(
    X_train_scaled, y_train_reg,
    eval_set=[(X_train_scaled, y_train_reg), (X_test_scaled, y_test_reg)],
    verbose=20
)
print("[SUCCESS] XGBoost Regressor training completed.")

print("\n[INFO] Tuning and training XGBoost Classifier...")
xgb_cls_grid = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]}
xgb_cls_search = GridSearchCV(xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42), xgb_cls_grid, cv=tscv, n_jobs=-1, verbose=1)
xgb_cls_search.fit(X_train_scaled, y_train_cls)

print(f"[INFO] Best XGBoost Classifier parameters: {xgb_cls_search.best_params_}")
print("[INFO] Fitting final XGBoost Classifier and displaying training rounds (epochs) and losses live...")
xgb_cls_model = xgb.XGBClassifier(
    n_estimators=xgb_cls_search.best_params_['n_estimators'],
    learning_rate=xgb_cls_search.best_params_['learning_rate'],
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

xgb_cls_model.fit(
    X_train_scaled, y_train_cls,
    eval_set=[(X_train_scaled, y_train_cls), (X_test_scaled, y_test_cls)],
    verbose=20
)
print("[SUCCESS] XGBoost Classifier training completed.")

joblib.dump(xgb_reg_model, 'models/xgboost_regressor.pkl')
joblib.dump(xgb_cls_model, 'models/xgboost_classifier.pkl')
print("[INFO] XGBoost models saved successfully inside models/")

In [ ]:
# ==========================================================================
# 3. MODEL TRAINING - SUPPORT VECTOR MACHINE (SVM - WITH OPTIMIZER SOLVER LOGS)
# ==========================================================================
from sklearn.svm import LinearSVR, SVC

# Downsample representing globally for ultra-fast, responsive training iterations
X_train_svr_down = X_train_scaled[::25]
y_train_reg_svr_down = y_train_reg_svr[::25]
X_train_svc_down = X_train_scaled[::25]
y_train_cls_down = y_train_cls[::25]

print(f"[INFO] Tuning SVM LinearSVR on representational pool ({len(X_train_svr_down)} samples)...")
svr_grid = {'C': [0.1, 1.0, 10.0], 'epsilon': [0.0, 0.1]}
svr_search = GridSearchCV(LinearSVR(max_iter=5000, random_state=42), svr_grid, cv=tscv, n_jobs=-1, verbose=1)
svr_search.fit(X_train_svr_down, y_train_reg_svr_down)

print(f"[INFO] Best SVR parameters: {svr_search.best_params_}")
print("[INFO] Training final SVM LinearSVR model with solver iteration logs enabled...")
svr_model = LinearSVR(
    C=svr_search.best_params_['C'],
    epsilon=svr_search.best_params_['epsilon'],
    max_iter=5000,
    random_state=42,
    verbose=1
)
svr_model.fit(X_train_svr_down, y_train_reg_svr_down)
print("[SUCCESS] SVM LinearSVR training completed.")

print(f"\n[INFO] Tuning SVM SVC on representational pool ({len(X_train_svc_down)} samples)...")
svc_grid = {'C': [1.0, 10.0], 'kernel': ['rbf']}
svc_search = GridSearchCV(SVC(probability=True, random_state=42), svc_grid, cv=tscv, n_jobs=-1, verbose=1)
svc_search.fit(X_train_svc_down, y_train_cls_down)

print(f"[INFO] Best SVC parameters: {svc_search.best_params_}")
print("[INFO] Training final SVM SVC Classifier model with solver iteration logs enabled...")
svc_model = SVC(
    C=svc_search.best_params_['C'],
    kernel=svc_search.best_params_['kernel'],
    probability=True,
    random_state=42,
    verbose=True
)
svc_model.fit(X_train_svc_down, y_train_cls_down)
print("[SUCCESS] SVM SVC training completed.")

joblib.dump(svr_model, 'models/svm_regressor.pkl')
joblib.dump(svc_model, 'models/svm_classifier.pkl')
joblib.dump(scaler, 'models/svm_scaler.pkl')
joblib.dump(svr_target_scaler, 'models/svm_target_scaler.pkl')
print("[INFO] SVM models and scalers saved successfully inside models/")

In [ ]:
# ==========================================================================
# UNIFIED MODEL COMPARATIVE EVALUATION
# ==========================================================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("[INFO] Generating predictions for out-of-sample test split...")

# Regression model evaluations
y_pred_rf_reg = rf_reg_model.predict(X_test_scaled)
y_pred_xgb_reg = xgb_reg_model.predict(X_test_scaled)

# SVM regressor re-scaled dynamic prediction
y_pred_svr_scaled = svr_model.predict(X_test_scaled)
prices_hist = test_df['Close'].values
active_mean = np.mean(prices_hist)
active_std = max(np.std(prices_hist), active_mean * 0.01)
y_pred_svr_reg = y_pred_svr_scaled * active_std + active_mean

# Classification model evaluations
y_pred_rf_cls = rf_cls_model.predict(X_test_scaled)
y_pred_xgb_cls = xgb_cls_model.predict(X_test_scaled)
y_pred_svc_cls = svc_model.predict(X_test_scaled)

# Regression evaluation metrics
reg_models = ['Random Forest Regressor', 'XGBoost Regressor', 'SVM Regressor']
maes = [
    mean_absolute_error(y_test_reg, y_pred_rf_reg),
    mean_absolute_error(y_test_reg, y_pred_xgb_reg),
    mean_absolute_error(y_test_reg, y_pred_svr_reg)
]
mses = [
    mean_squared_error(y_test_reg, y_pred_rf_reg),
    mean_squared_error(y_test_reg, y_pred_xgb_reg),
    mean_squared_error(y_test_reg, y_pred_svr_reg)
]
rmses = [np.sqrt(m) for m in mses]
r2s = [
    r2_score(y_test_reg, y_pred_rf_reg),
    r2_score(y_test_reg, y_pred_xgb_reg),
    r2_score(y_test_reg, y_pred_svr_reg)
]

print("\n--- Regression Performance Metrics ---")
for name, mae, mse, rmse, r2 in zip(reg_models, maes, mses, rmses, r2s):
    print(f"{name:25}: MAE={mae:.4f}, MSE={mse:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

# Classification evaluation metrics
cls_models = ['Random Forest Classifier', 'XGBoost Classifier', 'SVM Classifier']
y_preds_cls = [y_pred_rf_cls, y_pred_xgb_cls, y_pred_svc_cls]

print("\n--- Classification Performance Metrics ---")
for name, y_p in zip(cls_models, y_preds_cls):
    acc = accuracy_score(y_test_cls, y_p)
    prec = precision_score(y_test_cls, y_p, zero_division=0)
    rec = recall_score(y_test_cls, y_p, zero_division=0)
    f1 = f1_score(y_test_cls, y_p, zero_division=0)
    print(f"{name:25}: Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1-Score={f1:.4f}")

print("\n[SUCCESS] Unified comparative evaluation completed successfully.")